# `models` -- Model Zoo Builders

Two factory functions, `make_zoo()` and `make_score_zoo()`, each returning a **fresh dict of untrained scikit-learn models**. Centralising the model definitions here (rather than constructing them ad hoc wherever they're used) means every training call in `pipeline.py` -- pre-match, dynamic 2nd innings, dynamic 1st innings, phase-specific -- uses the *exact same* hyperparameters, which is what makes their results comparable to each other.

**Design decisions worth calling out:**
- Every classifier is wrapped in `CalibratedClassifierCV`   (isotonic regression, 3-fold CV). This means `predict_proba()`   output is a genuinely calibrated probability, not just a raw   model score -- essential given this whole project treats   calibration as a first-class metric (see `metrics.py`).
- The `'XGBoost'` dict key uses **`sklearn.HistGradientBoostingClassifier`**,   not the native `xgboost` package. This is a deliberate   substitution: native XGBoost requires Homebrew's `libomp` on   macOS, which isn't always available (this bit the original   Phase-1 investigation directly). HistGradientBoosting is a   very similar gradient-boosting algorithm with zero extra   system dependencies, so results stay reproducible everywhere.
- SVM uses `LinearSVC`/`LinearSVR` (linear kernel only), not the   full-kernel `SVC`/`SVR` -- the delivery-level dataset has   ~95,000+ rows, and a full-kernel SVM's training time scales   roughly quadratically with sample count, making it   impractically slow at this size.
- `SEED = 42` is set once at module level and threaded through   every `random_state` argument, so every run of this pipeline   is fully reproducible.

In [1]:
SEED: int = 42

from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import (
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import LinearSVC, LinearSVR

## `make_zoo()` -- classification models (win probability)

Five calibrated classifiers, all trained on the same feature matrix in `pipeline.py`'s pre-match and dynamic 2nd/1st-innings functions:

| Key | Base estimator | Hyperparameters |
|---|---|---|
| `Logistic` | `LogisticRegression` | `C=1.0, max_iter=500` |
| `Random Forest` | `RandomForestClassifier` | `n_estimators=100, max_depth=6` |
| `Gradient BT` | `GradientBoostingClassifier` | `n_estimators=100, max_depth=3` |
| `XGBoost` | `HistGradientBoostingClassifier` (see module docstring) | `max_iter=100, max_depth=4` |
| `SVM` | `LinearSVC` | `C=1.0, max_iter=3000` |

**Important usage note** (from the docstring): call this function fresh every time you need untrained models for a new train/test split -- never reuse a dict returned by a previous call, since those models will already be fitted to a different split's data.

In [2]:
def make_zoo() -> dict:
    """
    Return a fresh dict of five calibrated classifiers for win probability.

    Keys match the display names used in all result tables and figures.
    Call this function each time you need a new set of untrained models
    (do not reuse a trained zoo for a new split).
    """
    return {
        "Logistic": CalibratedClassifierCV(
            LogisticRegression(C=1.0, max_iter=500, random_state=SEED),
            method="isotonic",
            cv=3,
        ),
        "Random Forest": CalibratedClassifierCV(
            RandomForestClassifier(n_estimators=100, max_depth=6, random_state=SEED),
            method="isotonic",
            cv=3,
        ),
        "Gradient BT": CalibratedClassifierCV(
            GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=SEED),
            method="isotonic",
            cv=3,
        ),
        "XGBoost": CalibratedClassifierCV(
            HistGradientBoostingClassifier(max_iter=100, max_depth=4, random_state=SEED),
            method="isotonic",
            cv=3,
        ),
        "SVM": CalibratedClassifierCV(
            LinearSVC(C=1.0, max_iter=3000, random_state=SEED),
            method="isotonic",
            cv=3,
        ),
    }

## `make_score_zoo()` -- regression models (final score prediction)

Five regressors, evaluated in `pipeline.py::train_score_zoo_and_save` at multiple horizons (pre-match, and at various overs into an innings) predicting the batting team's final score in runs:

| Key | Base estimator | Hyperparameters |
|---|---|---|
| `Linear` | `Ridge` | `alpha=1.0` |
| `Random Forest` | `RandomForestRegressor` | `n_estimators=100, max_depth=8` |
| `Gradient BT` | `GradientBoostingRegressor` | `n_estimators=100, max_depth=4` |
| `XGBoost` | `HistGradientBoostingRegressor` | `max_iter=100, max_depth=5` |
| `SVR` | `LinearSVR` | `C=1.0, max_iter=3000` |

No calibration wrapper here -- calibration (in the probability-calibration sense) only applies to classifiers; regression quality is judged directly by MAE/RMSE/R² instead (see `docs/known_limitations.md` for why pre-match R² is expected to be negative).

In [3]:
def make_score_zoo() -> dict:
    """
    Return a fresh dict of five regressors for final innings score prediction.

    Evaluated at over horizons 5, 10, 15, 18 and pre-match.
    Metric: MAE (runs), RMSE, R².
    """
    return {
        "Linear": Ridge(alpha=1.0),
        "Random Forest": RandomForestRegressor(
            n_estimators=100, max_depth=8, random_state=SEED
        ),
        "Gradient BT": GradientBoostingRegressor(
            n_estimators=100, max_depth=4, random_state=SEED
        ),
        "XGBoost": HistGradientBoostingRegressor(
            max_iter=100, max_depth=5, random_state=SEED
        ),
        "SVR": LinearSVR(C=1.0, max_iter=3000),
    }